# Projeto Fictus | Análise Logística — Bloco 3: Impacto no SLA e na Satisfação do Cliente

---

## Pergunta Central do Bloco
> **A internalização entrega SLA melhor — ou apenas troca um problema por outro?**

---

## Contexto do Bloco

A internalização só se justifica se entregar uma qualidade superior à terceirizada. Este bloco investiga se o controle direto sobre a operação se traduz em redução real de prazos e aumento da satisfação do cliente.

Exploramos a correlação entre a eficiência logística e a percepção de valor da marca. Para o analista, o foco aqui é validar se a promessa de melhoria de serviço é financeiramente relevante ou se é apenas uma troca de um problema por outro sob nova gestão.


**Este bloco investiga:**
1. Quanto vale financeiramente cada ponto percentual de melhoria de SLA?
2. A internalização entregaria controle real sobre o lead time ou apenas transfere o problema?
3. Quais rotas têm maior potencial de ganho de SLA com operação própria?
4. A melhoria de SLA teria impacto diferente por categoria de produto — produtos de maior ticket se beneficiam mais ou menos da melhoria de prazo?

---


## Configuração

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
import matplotlib.ticker as mticker, matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
import warnings
from pathlib import Path
try:
    _base = Path(__file__).resolve().parent
except NameError:
    _base = Path().resolve()
def _find_base(start: Path) -> Path:
    for p in [start, start.parent, start.parent.parent]:
        if (p / "data").exists() or (p / "notebooks").exists():
            return p
    return start
BASE_DIR = _find_base(_base)
DIR_LOG  = BASE_DIR / "data" / "logistics"
DIR_EXPORTS = BASE_DIR / "exports"
DIR_EXPORTS.mkdir(parents=True, exist_ok=True)
warnings.filterwarnings("ignore")
COR_FRETE="#C0392B"; COR_RECEITA="#1B4F72"; COR_MARGEM="#27AE60"
COR_ALERTA="#E74C3C"; COR_NEUTRO="#7F8C8D"; COR_DESTAQUE="#E67E22"; COR_ROXO="#8E44AD"
sns.set_theme(style="whitegrid", font_scale=1.0)
plt.rcParams.update({"figure.dpi":150,"savefig.dpi":150,"savefig.bbox":"tight",
    "font.family":"sans-serif","axes.spines.top":False,"axes.spines.right":False})
def fmt_pct(x,pos=None): return f"{x:.1f}%"
def salvar(fig,nome):
    caminho = DIR_EXPORTS/f"{nome}.png"; fig.savefig(caminho); print(f"  -> Salvo: {caminho.name}")
print("Ambiente configurado.")


## Carregamento

In [ ]:
def ler(f,**kw):
    df=pd.read_csv(DIR_LOG/f,low_memory=False,**kw); df.columns=df.columns.str.strip(); return df
log_fato = ler("log_fato.csv"); log_rota = ler("log_rota.csv")
log_mensal = ler("log_mensal.csv"); log_trim = ler("log_trimestral.csv")
for col in ["preco","valor_frete","lead_time_dias","atraso_dias","nota_review","entregue_no_prazo"]:
    if col in log_fato.columns: log_fato[col] = pd.to_numeric(log_fato[col], errors="coerce")
# Premissas de melhoria de SLA (auditaveis)
MELHORIA_SLA_PP   = 8    # pontos percentuais de ganho de SLA com operacao propria
REDUCAO_FRETE_CLI = 0.20 # reducao no frete ao cliente
AUMENTO_VOLUME    = 0.10 # aumento de pedidos por menor frete
periodos_ord = sorted(log_fato["periodo"].dropna().unique())
print(f"Dados: {len(log_fato):,} pedidos | {periodos_ord[0]} a {periodos_ord[-1]}")


---

## Análise 1 — Quanto vale financeiramente cada pp de melhoria de SLA?

> *"A Análise de Vendas estabeleceu que cada dia adicional de atraso reduz a nota de review de forma mensurável. A correlação observada é usada aqui para estimar o valor financeiro de melhorar o SLA em 1, 5 ou 10 pontos percentuais — transformando um argumento de qualidade em argumento econômico."*

**Framework:** Correlação de Pearson + Análise de causa e efeito
**Entrega:** Tabela de valor financeiro por pp de melhoria de SLA em diferentes cenários de volume

**Como este script responde à pergunta:**
> O script recalcula a correlação lead time × nota de review e usa a inclinação da regressão para estimar o impacto de cada ponto percentual de melhoria de SLA na satisfação média. Em seguida, converte essa melhoria de satisfação em receita adicional usando a elasticidade implícita declarada como premissa: melhor nota → maior propensão de recompra → maior receita por cliente.
>
> 1. **Correlação lead time × nota (regressão):** Confirma o coeficiente de impacto por dia de atraso. A linha de regressão e os intervalos de confiança mostram se a relação é robusta ou tem muita variabilidade.
> 2. **Tabela de valor por pp de SLA:** Para cada nível de melhoria (1pp, 5pp, 10pp), calcula a receita adicional projetada nos cenários atual e com crescimento de volume.

**Análise do Resultado:**
Esta análise revela o "lucro invisível" da logística. Ao melhorar o índice de entregas no prazo (SLA), não estamos apenas sendo eficientes, estamos recuperando receita que seria perdida por insatisfação ou cancelamentos. Traduzir pontos percentuais em valor monetário permite ao comprador entender que o investimento em qualidade logística tem um retorno direto no faturamento mensal.

In [ ]:
df_c = log_fato[["lead_time_dias","nota_review","entregue_no_prazo","preco"]].dropna()
r_lt_nota, p_lt_nota = stats.pearsonr(df_c["lead_time_dias"], df_c["nota_review"])
m_reg, b_reg, *_ = stats.linregress(df_c["lead_time_dias"], df_c["nota_review"])

nota_prazo  = df_c[df_c["entregue_no_prazo"]==1]["nota_review"].mean()
nota_atraso = df_c[df_c["entregue_no_prazo"]==0]["nota_review"].mean()
delta_nota  = nota_atraso - nota_prazo
ticket_m    = log_fato["preco"].mean()
n_ped_m     = log_mensal["n_pedidos"].mean()
# Elasticidade: cada 0.1 pt de nota = 1% de recompra (conservador)
ELAST_NOTA_VOL = 0.10

# Tabela de valor por pp de SLA
sla_atual = log_fato["entregue_no_prazo"].mean() * 100
melhorias = [1, 3, 5, 8, 10]
tabela_sla = []
for pp in melhorias:
    pedidos_add = n_ped_m * (pp / 100) * ELAST_NOTA_VOL * 10  # estimativa
    receita_add = pedidos_add * ticket_m
    tabela_sla.append({"Melhoria SLA": f"+{pp}pp", "SLA resultante": f"{sla_atual+pp:.1f}%",
                       "Pedidos adicionais/mes": f"{pedidos_add:,.0f}",
                       "Receita adicional/mes": f"R$ {receita_add:,.0f}"})
df_tabela = pd.DataFrame(tabela_sla)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Analise 1 - Valor Financeiro da Melhoria de SLA", fontsize=13, fontweight="bold")

# Regressao lead time x nota
sample = df_c.sample(min(3000, len(df_c)), random_state=42)
axes[0].scatter(sample["lead_time_dias"], sample["nota_review"],
                color=COR_FRETE, alpha=0.2, s=10)
xfit = np.linspace(df_c["lead_time_dias"].min(), df_c["lead_time_dias"].quantile(0.95), 100)
axes[0].plot(xfit, m_reg*xfit+b_reg, color=COR_RECEITA, linewidth=2)
axes[0].set_xlabel("Lead time (dias)")
axes[0].set_ylabel("Nota de review (1-5)")
axes[0].set_title(f"Lead Time x Nota\ncorr={r_lt_nota:.3f} | {m_reg:.4f} pts/dia | p={p_lt_nota:.4f}", fontsize=11)

# Bar de receita adicional por pp
receitas_add = [n_ped_m * (pp/100) * ELAST_NOTA_VOL * 10 * ticket_m for pp in melhorias]
bars = axes[1].bar([f"+{pp}pp" for pp in melhorias], [r/1000 for r in receitas_add],
                   color=[COR_MARGEM if r > 0 else COR_ALERTA for r in receitas_add], alpha=0.85)
for bar, val in zip(bars, receitas_add):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
                 f"R$ {val:,.0f}", ha="center", fontsize=8)
axes[1].set_xlabel("Melhoria de SLA (pp)")
axes[1].set_ylabel("Receita adicional mensal estimada (R$ mil)")
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f"R$ {v:,.0f}K"))
axes[1].set_title(f"Receita Adicional por Melhoria de SLA\n(SLA atual: {sla_atual:.1f}%)", fontsize=11)
plt.tight_layout()
salvar(fig, "10_valor_financeiro_sla")
plt.show()

print("Tabela de valor por melhoria de SLA:")
print(df_tabela.to_string(index=False))
print(f"\nSLA atual           : {sla_atual:.1f}%")
print(f"Com internalizacao  : {sla_atual + MELHORIA_SLA_PP:.1f}% (premissa: +{MELHORIA_SLA_PP}pp)")
receita_add_total = n_ped_m * (MELHORIA_SLA_PP/100) * ELAST_NOTA_VOL * 10 * ticket_m
print(f"Receita adicional/m : R$ {receita_add_total:,.0f} (premissa conservadora)")


---

## Análise 2 — A internalização permitiria controle real ou apenas transfere a variabilidade?

> *"Internalizar não elimina automaticamente a variabilidade — ela muda de mãos. Uma operação própria mal dimensionada pode ter SLA pior do que o modelo terceirizado. A decomposição do lead time em componente interno e externo determina se o gargalo está na preparação do pedido (controlável com operação própria) ou no transporte (dependente de terceiros independentemente do modelo)."*

**Framework:** Teoria das Restrições — localização do gargalo
**Entrega:** Decomposição da variabilidade do SLA atual: interna vs externa

**Como este script responde à pergunta:**
> O Olist registra dois timestamps que permitem decompor o lead time: `data_aprovacao` → `data_envio_transportadora` (tempo de preparação, responsabilidade interna) e `data_envio_transportadora` → `data_entrega_cliente` (tempo de transporte, responsabilidade da transportadora). O script calcula quanto de cada componente existe e onde a variabilidade é maior.
>
> 1. **Decomposição do lead time:** Barras empilhadas mostrando preparação vs transporte por período. Se a fatia de preparação for maior e mais variável, o gargalo é interno — resolvível com operação própria. Se for o transporte, a internalização pode não resolver.
> 2. **Variabilidade por componente:** Boxplot da distribuição de cada componente. Componente com caixa maior e outliers mais extremos é onde a variabilidade está concentrada.

**Análise do Resultado:**
Aqui investigamos a origem do atraso. Se o gargalo estiver na "preparação do pedido" (etapa interna), a internalização oferece controle total para resolver o problema. Se o atraso ocorrer apenas no transporte (etapa externa), a mudança de modelo teria um impacto menor. Este diagnóstico evita que a empresa invista em frota própria para tentar resolver um problema que, às vezes, é de processo administrativo.


In [ ]:
# Verifica disponibilidade das colunas de decomposicao
tem_decomp = all(c in log_fato.columns for c in
    ["data_compra","data_entrega_cliente","data_previsao_entrega"])
tem_fases  = all(c in log_fato.columns for c in
    ["data_aprovacao","data_envio_transportadora"])

if tem_fases:
    log_fato["data_aprovacao"] = pd.to_datetime(log_fato["data_aprovacao"], errors="coerce")
    log_fato["data_envio"]     = pd.to_datetime(log_fato["data_envio_transportadora"], errors="coerce")
    log_fato["t_preparacao"]   = (log_fato["data_envio"] - log_fato["data_aprovacao"]).dt.days
    log_fato["t_transporte"]   = (log_fato["data_entrega_cliente"] - log_fato["data_envio"]).dt.days
    log_fato["t_preparacao"]   = pd.to_numeric(log_fato["t_preparacao"], errors="coerce")
    log_fato["t_transporte"]   = pd.to_numeric(log_fato["t_transporte"], errors="coerce")
    tem_decomp = True
    prep_m  = log_fato["t_preparacao"].mean()
    trans_m = log_fato["t_transporte"].mean()
    lead_m  = log_fato["lead_time_dias"].mean()
    pct_prep  = prep_m / lead_m * 100 if lead_m > 0 else 50
    pct_trans = trans_m / lead_m * 100 if lead_m > 0 else 50
    gargalo = "PREPARACAO INTERNA" if prep_m > trans_m else "TRANSPORTE (terceirizado)"
else:
    # Estimativa proporcional quando colunas nao disponiveis
    lead_m  = log_fato["lead_time_dias"].mean()
    prep_m  = lead_m * 0.35   # estimativa: ~35% preparacao
    trans_m = lead_m * 0.65   # estimativa: ~65% transporte
    pct_prep  = 35.0; pct_trans = 65.0
    gargalo = "NAO DETERMINADO — estimativa: transporte (65%)"
    print("[AVISO] Colunas data_aprovacao/data_envio_transportadora nao disponiveis.")
    print("        Usando estimativa proporcional para decomposicao do lead time.")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Analise 2 - Decomposicao do Lead Time: Interno vs Externo", fontsize=13, fontweight="bold")

# Barras de proporcao
comps = ["Preparacao\n(interno)", "Transporte\n(terceirizado)", "Lead Time\nTotal"]
vals  = [prep_m, trans_m, lead_m]
cores_c = [COR_RECEITA, COR_FRETE, COR_NEUTRO]
bars = axes[0].bar(comps, vals, color=cores_c, alpha=0.85)
for bar, val in zip(bars, vals):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.1,
                 f"{val:.1f}d", ha="center", fontsize=10, fontweight="bold")
axes[0].set_ylabel("Dias medios")
axes[0].set_title(f"Decomposicao Media do Lead Time\nGargalo: {gargalo}", fontsize=11)

# Pizza de proporcao
axes[1].pie([pct_prep, pct_trans],
            labels=[f"Preparacao interna\n({pct_prep:.1f}%)", f"Transporte externo\n({pct_trans:.1f}%)"],
            colors=[COR_RECEITA, COR_FRETE], autopct="%1.0f%%", startangle=90)
recuperavel = pct_prep  # % do lead time que operacao propria pode controlar
axes[1].set_title(f"Proporcao de Cada Etapa no Lead Time\n{recuperavel:.0f}% controlavel com operacao propria", fontsize=11)

plt.tight_layout()
salvar(fig, "11_decomposicao_lead_time_interno_externo")
plt.show()

print(f"Lead time total medio   : {lead_m:.1f} dias")
print(f"  Preparacao (interno)  : {prep_m:.1f} dias ({pct_prep:.1f}%)")
print(f"  Transporte (externo)  : {trans_m:.1f} dias ({pct_trans:.1f}%)")
print(f"Gargalo principal       : {gargalo}")
print(f"% controlavel internamente: {pct_prep:.1f}%")
print(f"Conclusao: a internalizacao tem {'ALTO' if pct_prep > 40 else 'MODERADO' if pct_prep > 25 else 'BAIXO'} potencial de melhoria de SLA")


---

## Análise 3 — Quais rotas têm maior potencial de ganho de SLA com operação própria?

> *"Nem todas as rotas se beneficiam igualmente da internalização. O cruzamento do SLA atual por região com o volume de receita associado identifica onde a melhoria de qualidade teria maior impacto financeiro — e define as rotas prioritárias para o modelo híbrido."*

**Framework:** Pareto + Análise de causa e efeito
**Entrega:** Ranking de rotas por potencial de ganho de SLA × receita associada

**Como este script responde à pergunta:**
> O script cria um score de potencial de ganho para cada rota: combina o quanto o SLA está abaixo da meta (gap a recuperar) com o peso da rota na receita total (impacto financeiro). Rotas com grande gap de SLA e alta receita associada são as candidatas prioritárias à internalização no modelo híbrido.
>
> 1. **Scatter receita × gap de SLA:** Cada ponto é uma rota. Rotas no quadrante superior direito (alta receita + grande gap de SLA) são as de maior potencial de impacto financeiro com melhoria de qualidade.
> 2. **Ranking de potencial de ganho:** Lista as rotas ordenadas pelo score combinado, com o valor financeiro estimado da melhoria de SLA em cada uma.

**Análise do Resultado:**
Identificamos as "rotas de oportunidade". Em vez de internalizar toda a logística de forma indiscriminada, mapeamos onde a operação própria causaria o maior salto de qualidade imediato. Para o investidor, este é o guia de onde começar a operação para gerar o maior impacto positivo na percepção do cliente com o menor custo inicial.

In [ ]:
# Score de potencial: gap de SLA x receita
sla_meta = 90.0
lr = log_rota.copy()
lr["gap_sla"]       = (sla_meta - lr["pct_no_prazo"].fillna(sla_meta)).clip(lower=0)
lr["score_potencial"]= (lr["gap_sla"] / sla_meta) * (lr["pct_receita"] / lr["pct_receita"].max())
lr_top = lr.nlargest(20, "receita_total")

# Valor financeiro do ganho de SLA por rota
receita_add_por_pp = log_mensal["n_pedidos"].mean() * ELAST_NOTA_VOL * 0.10 * log_fato["preco"].mean()
lr["receita_add_est"] = lr["gap_sla"] * receita_add_por_pp * (lr["pct_receita"] / 100)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Analise 3 - Potencial de Ganho de SLA por Rota", fontsize=13, fontweight="bold")

# Scatter
cores_sc = ["#E74C3C" if s > lr_top["score_potencial"].quantile(0.7) else "#1B4F72"
            for s in lr_top["score_potencial"]]
scatter = axes[0].scatter(
    lr_top["pct_receita"], lr_top["gap_sla"],
    s=lr_top["receita_total"]/lr_top["receita_total"].max()*600+30,
    c=cores_sc, alpha=0.75, edgecolors="white", linewidth=0.5
)
axes[0].axhline(0, color="black", linewidth=0.8)
axes[0].axvline(lr_top["pct_receita"].median(), color=COR_NEUTRO, linestyle="--", linewidth=1, alpha=0.5)
for _, row in lr_top.nlargest(6,"score_potencial").iterrows():
    axes[0].annotate(row["rota"][:15], (row["pct_receita"], row["gap_sla"]),
                     fontsize=6, xytext=(2,2), textcoords="offset points")
axes[0].set_xlabel("% da Receita Total")
axes[0].set_ylabel(f"Gap de SLA em relacao a meta de {sla_meta:.0f}% (pp)")
axes[0].set_title("Receita x Gap de SLA\n(tamanho = receita | vermelho = alta prioridade)", fontsize=11)

# Ranking
top10_pot = lr.nlargest(10, "score_potencial").sort_values("score_potencial")
axes[1].barh([r[:22] for r in top10_pot["rota"]], top10_pot["receita_add_est"]/1000,
             color=COR_FRETE, alpha=0.85)
for i, (_, row) in enumerate(top10_pot.iterrows()):
    axes[1].text(row["receita_add_est"]/1000+0.1, i,
                 f"  SLA:{row['pct_no_prazo']:.0f}% | gap:{row['gap_sla']:.1f}pp",
                 va="center", fontsize=7)
axes[1].set_xlabel("Receita adicional estimada/mes (R$ mil)")
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f"R$ {v:,.0f}K"))
axes[1].set_title("Top 10 Rotas por Potencial de Ganho", fontsize=11)
plt.tight_layout()
salvar(fig, "12_potencial_ganho_sla_por_rota")
plt.show()

top3 = lr.nlargest(3,"score_potencial")
print("Top 3 rotas por potencial de ganho:")
for _,r in top3.iterrows():
    print(f"  {r['rota']:<30} SLA:{r['pct_no_prazo']:.0f}% gap:{r['gap_sla']:.1f}pp receita:{r['pct_receita']:.1f}% pot_receita:R${r['receita_add_est']:,.0f}/m")


---

## Análise 4 — A melhoria de SLA teria impacto diferente por categoria — produtos de maior valor são mais sensíveis a atrasos?

> *"Um dia de atraso numa entrega de R$ 20 tem impacto diferente de um atraso numa entrega de R$ 500. Produtos de alto valor têm clientes com maior expectativa de serviço e menor tolerância a falha logística — o que significa que o ganho financeiro de melhorar o SLA não é distribuído uniformemente entre categorias. Identificar onde o SLA importa mais é onde o ROI da internalização é maior."*

**Framework:** Análise de elasticidade SLA × satisfação por segmento de valor  
**Entrega:** Mapa de sensibilidade ao SLA por categoria (ticket × impacto na nota) + ranking de ROI de SLA por categoria

**Como este script responde à pergunta:**
> A distribuição do impacto do SLA por categoria é o que transforma a análise de logística em uma decisão de negócio precisa. Este script constrói duas visualizações:
>
> 1. **Mapa de sensibilidade ticket × impacto na nota:** Para cada categoria, calcula a correlação entre lead time e nota de review (quanto o atraso penaliza a satisfação) e cruza com o ticket médio. O quadrante superior direito — alto ticket e alta sensibilidade ao atraso — é onde a melhoria de SLA gera mais valor por pedido. Categorias nesse quadrante são as candidatas prioritárias para rotas de entrega expressa dentro do modelo híbrido.
> 2. **Ranking de ROI de SLA por categoria:** Combina a sensibilidade ao SLA com o volume de pedidos e o ticket médio para estimar o ganho financeiro mensal esperado de melhorar o SLA em cada categoria. A fórmula é direta: mais pedidos × maior ticket × maior sensibilidade = maior ROI de investimento em SLA. Categorias com ROI alto e SLA atual ruim são onde a internalização teria impacto mais imediato e mensurável.

**Análise do Resultado:**
Nem todo produto exige a mesma urgência. Esta análise segmenta o impacto por categoria para entender onde o cliente é mais sensível ao prazo. Descobrir que categorias de alto ticket (mais caras) são mais afetadas por atrasos ajuda a priorizar a logística própria para os produtos que trazem mais margem, otimizando o Retorno sobre o Investimento (ROI) por setor de estoque.

In [ ]:
# ─── Análise 4 — Sensibilidade ao SLA por Categoria ──────────────────────────

# Calcular por categoria: ticket médio, correlação SLA×nota, volume, SLA atual
cats_validas = (
    log_fato.groupby("nome_categoria_produto")["id_pedido"].nunique()
    .pipe(lambda s: s[s >= 50].index)
)
df_cat_sla = log_fato[log_fato["nome_categoria_produto"].isin(cats_validas)].copy()

def calc_cat_metrics(grp):
    d = grp.dropna(subset=["lead_time_dias", "nota_review", "entregue_no_prazo"])
    if len(d) < 20:
        return None
    r, p = stats.pearsonr(d["lead_time_dias"], d["nota_review"])
    return pd.Series({
        "ticket_medio":   grp["preco"].mean(),
        "n_pedidos":       len(grp),
        "sla_atual":       grp["entregue_no_prazo"].mean() * 100,
        "corr_lt_nota":    r,
        "sensibilidade":   abs(r),
        "p_valor":         p,
    })

cat_metrics = (
    df_cat_sla.groupby("nome_categoria_produto")
    .apply(calc_cat_metrics)
    .dropna()
)

# ROI de SLA: sensibilidade × ticket × volume × melhoria esperada
cat_metrics["roi_sla"] = (
    cat_metrics["sensibilidade"]
    * cat_metrics["ticket_medio"]
    * cat_metrics["n_pedidos"]
    * (MELHORIA_SLA_PP / 100)
)

# Quadrantes
med_ticket = cat_metrics["ticket_medio"].median()
med_sens   = cat_metrics["sensibilidade"].median()

def quadrante_cat(row):
    at = row["ticket_medio"]  >= med_ticket
    as_ = row["sensibilidade"] >= med_sens
    if at and as_:   return "Prioridade máxima"
    if at and not as_: return "Alto valor, baixa sensibilidade"
    if not at and as_: return "Baixo valor, alta sensibilidade"
    return "Baixa prioridade"

cat_metrics["quadrante"] = cat_metrics.apply(quadrante_cat, axis=1)
cores_quad = {
    "Prioridade máxima":            COR_ALERTA,
    "Alto valor, baixa sensibilidade": COR_DESTAQUE,
    "Baixo valor, alta sensibilidade": COR_ROXO,
    "Baixa prioridade":             COR_NEUTRO,
}

top10_roi = cat_metrics.nlargest(10, "roi_sla").sort_values("roi_sla")

# ─── Plot ─────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle("Análise 4 — Sensibilidade ao SLA por Categoria: Onde o Atraso Custa Mais",
             fontsize=13, fontweight="bold")

# Mapa de sensibilidade
for quad, cor in cores_quad.items():
    mask = cat_metrics["quadrante"] == quad
    axes[0].scatter(
        cat_metrics.loc[mask, "ticket_medio"],
        cat_metrics.loc[mask, "sensibilidade"],
        s=cat_metrics.loc[mask, "n_pedidos"] / cat_metrics["n_pedidos"].max() * 400 + 30,
        color=cor, alpha=0.75, label=quad, edgecolors="white", linewidth=0.5
    )
# Anotar top 5 por ROI
for cat, row in cat_metrics.nlargest(5, "roi_sla").iterrows():
    axes[0].annotate(cat[:18], (row["ticket_medio"], row["sensibilidade"]),
                     fontsize=7, xytext=(3, 3), textcoords="offset points")
axes[0].axhline(med_sens,   color="gray", linestyle="--", linewidth=0.8, alpha=0.6)
axes[0].axvline(med_ticket, color="gray", linestyle="--", linewidth=0.8, alpha=0.6)
axes[0].set_xlabel("Ticket médio da categoria (R$)")
axes[0].set_ylabel("|Correlação| lead time × nota de review")
axes[0].set_title("Mapa de Sensibilidade ao SLA\n(tamanho = volume de pedidos)", fontsize=11)
axes[0].legend(frameon=False, fontsize=8)
for spine in ["top", "right"]:
    axes[0].spines[spine].set_visible(False)

# Ranking de ROI de SLA
cores_roi = [cores_quad[q] for q in top10_roi["quadrante"]]
bars = axes[1].barh(
    [c[:25] for c in top10_roi.index],
    top10_roi["roi_sla"] / 1000,
    color=cores_roi, alpha=0.85
)
for bar, (cat, row) in zip(bars, top10_roi.iterrows()):
    axes[1].text(
        bar.get_width() + 0.2, bar.get_y() + bar.get_height() / 2,
        f"SLA:{row['sla_atual']:.0f}% | ticket:R${row['ticket_medio']:.0f}",
        va="center", fontsize=7
    )
axes[1].set_xlabel("ROI estimado de melhoria de SLA (R$ mil/mês)")
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"R${v:.0f}K"))
axes[1].set_title("Top 10 Categorias por ROI de Melhoria de SLA", fontsize=11)
for spine in ["top", "right"]:
    axes[1].spines[spine].set_visible(False)

plt.tight_layout()
salvar(fig, "13_sensibilidade_sla_categoria")
plt.show()

n_prior_max = (cat_metrics["quadrante"] == "Prioridade máxima").sum()
roi_total_est = cat_metrics["roi_sla"].sum() / 1000
top_cat = cat_metrics.nlargest(1, "roi_sla").iloc[0]

print("\n" + "="*55)
print("INSIGHT — SENSIBILIDADE AO SLA POR CATEGORIA")
print("="*55)
print(f"Categorias analisadas           : {len(cat_metrics)}")
print(f"Categorias de prioridade máxima : {n_prior_max}")
print(f"ROI total estimado de +{MELHORIA_SLA_PP}pp SLA : R$ {roi_total_est:,.0f}K/mês")
print(f"Categoria de maior ROI          : {top_cat.name[:35]}")
print(f"  SLA atual: {top_cat['sla_atual']:.1f}% | ticket: R${top_cat['ticket_medio']:.0f} | ROI: R${top_cat['roi_sla']:,.0f}/mês")


---
## Síntese do Bloco 3 — SLA e Satisfação do Cliente

> **Limitações desta análise:** a conversão de melhoria de SLA em receita adicional usa elasticidade de recompra declarada como premissa conservadora — não há modelo empírico validando essa relação no dataset. A relação entre lead time e nota de review é uma associação estatística; variáveis não observadas (categoria, ticket, região, perfil do seller) podem explicar parte da variação sem que o SLA seja o driver causal. A decomposição do lead time em componente interno e externo depende da disponibilidade dos timestamps de aprovação e envio — quando ausentes, utiliza-se estimativa proporcional declarada no código.


In [ ]:

# Sensibilidade por categoria (análise 4)
_n_prior_max_v = (cat_metrics["quadrante"] == "Prioridade máxima").sum()
_roi_total_v   = cat_metrics["roi_sla"].sum()
_top_cat_v     = cat_metrics.nlargest(1, "roi_sla").iloc[0]

_sla_atual  = log_fato["entregue_no_prazo"].mean() * 100
_sla_proj   = _sla_atual + MELHORIA_SLA_PP
_rec_add_sla= n_ped_m * (MELHORIA_SLA_PP/100) * ELAST_NOTA_VOL * 10 * ticket_m if 'n_ped_m' in dir() else log_mensal["n_pedidos"].mean() * (MELHORIA_SLA_PP/100) * 0.10 * 10 * log_fato["preco"].mean()
_gap_medio  = lr["gap_sla"].mean()
_pot_gargalo= pct_prep

s_sla     = "SLA ABAIXO DA META DE 90%" if _sla_atual < 90 else "SLA DENTRO DO BENCHMARK"
s_gargalo = f"GARGALO INTERNO ({pct_prep:.0f}% do lead time) — internalizacao pode resolver" if pct_prep > 35 else f"GARGALO EXTERNO ({pct_trans:.0f}% no transporte) — internalizacao tem impacto limitado"
s_rotas   = f"{len(lr[lr['gap_sla']>5])} rotas com gap SLA acima de 5pp — candidatas prioritarias"

n_alertas = sum([_sla_atual < 90, pct_prep <= 35])
if n_alertas == 0:
    sinal = "SLA COM ALTO POTENCIAL DE MELHORIA — internalizacao entregaria ganho real"
elif n_alertas == 1:
    sinal = "SLA COM POTENCIAL MODERADO — ganho de qualidade parcialmente dependente de gargalo externo"
else:
    sinal = "SLA COM BAIXO POTENCIAL INTERNO — verificar se gargalo esta na transportadora"

print("=" * 65)
print("SINTESE - BLOCO 3: IMPACTO NO SLA E NA SATISFACAO")
print("=" * 65)
print("\n[ SLA ATUAL ]")
print(f"  SLA atual               : {_sla_atual:.1f}% — {s_sla}")
print(f"  SLA com internalizacao  : {_sla_proj:.1f}% (+{MELHORIA_SLA_PP}pp — premissa)")
print(f"  Receita adicional est.  : R$ {_rec_add_sla:,.0f}/mes")
print("\n[ GARGALO ]")
print(f"  {s_gargalo}")
print("\n[ ROTAS PRIORITARIAS ]")
print(f"  {s_rotas}")
print("\n" + "=" * 65)
print("VEREDICTO PARCIAL - BLOCO 3")
print("=" * 65)
print(f"\nSinal geral: {sinal}")
print(f"\n[ SENSIBILIDADE POR CATEGORIA ]")
print(f"  Categorias prioridade máxima  : {_n_prior_max_v}")
print(f"  ROI total est. de +{MELHORIA_SLA_PP}pp SLA   : R$ {_roi_total_v:,.0f}/mês")
print(f"  Categoria de maior ROI        : {_top_cat_v.name[:30]}")
print("\nProximo passo: Bloco 4 - escalabilidade e riscos estruturais.")


---
*Próximo notebook: `04_escalabilidade_riscos.ipynb` — Qual modelo sustenta crescimento sem fragilizar a operação?*

> Esta análise faz parte do **Projeto Fictus**, conduzido pela Lufi Data Consulting. Os três módulos analíticos — Vendas, Logística e Finanças — compõem a base do Relatório de Recomendação de Aquisição.
